In [10]:
import numpy as np
import json
from torch.utils.data.dataset import TensorDataset
import  torch

In [15]:
with open("../activations/imagenet_train_hf/config.json") as f:
    config = json.load(f)


test = np.memmap(
    "../activations/imagenet_train_hf/embeddings.npy",
    dtype=np.float16,
    mode="r",
    shape=(config["dataset_size"], config["clip_embedding_size"]),
)
labels = np.memmap(
    "../activations/imagenet_train_hf/labels.npy",
    dtype=np.int64,
    mode="r",
    shape=(config["dataset_size"],),
)


dataset = TensorDataset(torch.from_numpy(test), torch.from_numpy(labels))


In [20]:
from torch.utils.data.dataloader import DataLoader

dl = DataLoader(dataset, batch_size=8)

for batch in dl:
    tensors, labels = batch

    print(tensors.shape)
    print(labels.shape)
    break

torch.Size([8, 512])
torch.Size([8])


In [23]:
from ca_sae.sae.batch_top_k import BatchTopKSAE

sae = BatchTopKSAE.from_pretrained(
    "../checkpoints/test/BatchTopKSAEFirst/ae.pt", k=64, device="cuda"
)

In [32]:
for batch in dl:
    x = batch[0].to("cuda")
    f = sae.encode(x)

    x_hat = sae.decode(f)

    loss = (x - x_hat)**2
    # loss = torch.nn.functional.mse_loss(x_hat, x)

    print(loss)
    break

tensor([[1.1086e-03, 5.2704e-03, 9.0683e-03,  ..., 3.9293e-03, 2.0399e-03,
         2.6667e-02],
        [8.2772e-04, 8.4545e-02, 2.2840e-02,  ..., 1.0566e-03, 2.6218e-04,
         7.7541e-05],
        [6.1778e-03, 1.4070e-02, 1.3940e-03,  ..., 1.0446e-03, 9.9758e-03,
         3.1786e-03],
        ...,
        [1.4884e-03, 9.6716e-03, 5.0570e-03,  ..., 5.5749e-04, 2.7411e-03,
         3.1027e-03],
        [4.0905e-02, 3.9162e-02, 4.5243e-02,  ..., 6.4301e-02, 3.0533e-02,
         7.3943e-03],
        [2.1302e-02, 2.1648e-02, 1.1654e-02,  ..., 2.8811e-02, 8.7137e-03,
         1.2025e-02]], device='cuda:0', grad_fn=<PowBackward0>)


In [63]:
from lapsum.topk import soft_topk
import torch

x = torch.rand(size=(1, 4096)) * 10
x.requires_grad_(True)
k = torch.tensor([64.0]).unsqueeze(0)
k.requires_grad_(True)
alpha = torch.tensor([0.05])
alpha.requires_grad_(True)
selection = soft_topk(x, k, alpha)

for x in selection[0]:
    # if x > 1e-4:
    print(x)

tensor(2.4798e-18, grad_fn=<UnbindBackward0>)
tensor(1.7795e-34, grad_fn=<UnbindBackward0>)
tensor(0., grad_fn=<UnbindBackward0>)
tensor(0., grad_fn=<UnbindBackward0>)
tensor(0., grad_fn=<UnbindBackward0>)
tensor(2.3814e-37, grad_fn=<UnbindBackward0>)
tensor(0., grad_fn=<UnbindBackward0>)
tensor(0., grad_fn=<UnbindBackward0>)
tensor(9.8424e-14, grad_fn=<UnbindBackward0>)
tensor(0., grad_fn=<UnbindBackward0>)
tensor(0.0051, grad_fn=<UnbindBackward0>)
tensor(0., grad_fn=<UnbindBackward0>)
tensor(0., grad_fn=<UnbindBackward0>)
tensor(1.3880e-31, grad_fn=<UnbindBackward0>)
tensor(9.8582e-06, grad_fn=<UnbindBackward0>)
tensor(1.4013e-45, grad_fn=<UnbindBackward0>)
tensor(9.2190e-26, grad_fn=<UnbindBackward0>)
tensor(6.1767e-14, grad_fn=<UnbindBackward0>)
tensor(3.4482e-27, grad_fn=<UnbindBackward0>)
tensor(7.5670e-44, grad_fn=<UnbindBackward0>)
tensor(0., grad_fn=<UnbindBackward0>)
tensor(0., grad_fn=<UnbindBackward0>)
tensor(1.4379e-36, grad_fn=<UnbindBackward0>)
tensor(2.0777e-31, grad_fn

In [1]:
from ca_sae.sae.softsae import SoftSAE

sae  = SoftSAE(512, 4096, k=64, alpha=0.8)

sae.encode(torch.rand(size=(4, 512)), use_hard_topk=False)

NameError: name 'torch' is not defined

In [11]:
import  torch

linear = torch.nn.Sequential(torch.nn.Linear(10, 2))

linear[0].weight

Parameter containing:
tensor([[-0.0940,  0.1369, -0.2571, -0.0204, -0.2120, -0.2322,  0.1091,  0.0094,
          0.2384,  0.0445],
        [-0.1682,  0.1423,  0.1657, -0.2282,  0.1607, -0.3157, -0.0632,  0.1615,
         -0.2702, -0.0080]], requires_grad=True)

In [2]:
from datasets import load_dataset

In [4]:
from labels import IMAGENET2012_CLASSES

list(IMAGENET2012_CLASSES.values())

['tench, Tinca tinca',
 'goldfish, Carassius auratus',
 'great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias',
 'tiger shark, Galeocerdo cuvieri',
 'hammerhead, hammerhead shark',
 'electric ray, crampfish, numbfish, torpedo',
 'stingray',
 'cock',
 'hen',
 'ostrich, Struthio camelus',
 'brambling, Fringilla montifringilla',
 'goldfinch, Carduelis carduelis',
 'house finch, linnet, Carpodacus mexicanus',
 'junco, snowbird',
 'indigo bunting, indigo finch, indigo bird, Passerina cyanea',
 'robin, American robin, Turdus migratorius',
 'bulbul',
 'jay',
 'magpie',
 'chickadee',
 'water ouzel, dipper',
 'kite',
 'bald eagle, American eagle, Haliaeetus leucocephalus',
 'vulture',
 'great grey owl, great gray owl, Strix nebulosa',
 'European fire salamander, Salamandra salamandra',
 'common newt, Triturus vulgaris',
 'eft',
 'spotted salamander, Ambystoma maculatum',
 'axolotl, mud puppy, Ambystoma mexicanum',
 'bullfrog, Rana catesbeiana',
 'tree frog, tree-f